# OpenClaw → ComfyUI → Wan 2.2 TI2V 5B

**HP only:** buka notebook ini di Google Colab, pilih GPU, lalu Run All.

This uses local/open-source Wan 2.2 through ComfyUI. There are no video API credits. Free GPU providers can still impose runtime/quota limits.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt
!pip -q install requests websocket-client

## Download Wan 2.2 TI2V 5B
Official ComfyUI model layout: diffusion model + UMT5 text encoder + Wan 2.2 VAE.

In [ ]:
%cd /content/ComfyUI
!mkdir -p models/diffusion_models models/text_encoders models/vae
!wget -q --show-progress -O models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors
!wget -q --show-progress -O models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
!wget -q --show-progress -O models/vae/wan2.2_vae.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors
!ls -lh models/diffusion_models models/text_encoders models/vae

## Start ComfyUI

In [ ]:
import subprocess, time, requests, os, signal
%cd /content/ComfyUI
comfy = subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--enable-cors-header','*'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(15)
print('PID:', comfy.pid)
print(requests.get('http://127.0.0.1:8188/system_stats').status_code)

## Public URL for OpenClaw
Uses a Cloudflare quick tunnel. No API/video credits are required. Keep the Colab tab/session alive while generating.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import subprocess, re, time
cf = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = cf.stdout.readline()
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[^ ]+trycloudflare.com', line)
        if m: url = m.group(0); break
    time.sleep(1)
print('COMFYUI_BASE_URL =', url)
open('/content/COMFYUI_BASE_URL.txt','w').write(url or '')

## Test endpoint
Copy the printed `COMFYUI_BASE_URL` into your OpenClaw ComfyUI provider configuration.

OpenClaw should point to this URL; it can then submit ComfyUI workflow prompts and retrieve generated files.

In [ ]:
import requests, os
base = open('/content/COMFYUI_BASE_URL.txt').read().strip()
print('BASE:', base)
if base:
    r = requests.get(base + '/system_stats', timeout=30)
    print('HTTP', r.status_code)
    print(r.text[:500])

## Recommended starting generation
For a free GPU, start small: portrait 480×832, 49 frames, 16 fps. Once stable, increase duration/quality. Wan 2.2 TI2V 5B is the hybrid text+image-to-video model.